# Current version : 10.C (2025-07-23)

# Libraries and directory (always run)

In [1]:
### import necessary libraries
# import geopandas as gpd
# from IPython.display import display
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import math
import numpy as np
import os
from IPython.display import clear_output
import pandas as pd
import seaborn as sns
import scanpy as sc
from adjustText import adjust_text
from sklearn.cluster import KMeans
import seaborn as sns
import matplotlib.pyplot as plt
# from sklearn.metrics import adjusted_rand_score
# from sklearn.neighbors import NearestNeighbors
# from sklearn.neighbors import KNeighborsClassifier
import warnings
import importlib
from datetime import datetime
import anndata
import progressbar
today = datetime.today().strftime('%Y-%m-%d')

from module.config_local import dir_raw, dir_processed
import module.dataviz_analysis as da

pd.options.display.max_rows = 2000


# Ensure text remains editable in SVG

mpl.rcParams['svg.fonttype'] = 'none'  # 'none' = keep text as text objects


# Optional: improve SVG precision

mpl.rcParams['svg.hashsalt'] = ''  # consistent hashes for reproducibility

# warnings.filterwarnings("ignore") 
# sc.logging.print_header()
# sc.set_figure_params(facecolor="white", figsize=(4, 10))
# sc.settings.verbosity = 1 # errors (0), warnings (1), info (2), hints (3)
# plt.rcParams["font.family"] = "Arial"
# sns.set_style("white")


start_time = datetime.now()

mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color= ['#009736','#EE2A35',"#3f488aff",
                                                    "#f79a00ff", "#cf1100ff", "#81a5bfff",
                                                    "#f9bd00ff","#547200ff", "#bfd8cdff"]) 

def print_with_elapsed_time(message):
    elapsed_time = datetime.now() - start_time
    elapsed_seconds = elapsed_time.total_seconds()
    print(f"[{elapsed_seconds:.2f} seconds] {message}")

In [2]:
# print(f"geopandas version: {gpd.__version__}")
print(f"pandas version: {importlib.metadata.version('pandas')}")
print(f"scanpy version: {importlib.metadata.version('scanpy')}")
print(f"matplotlib version: {importlib.metadata.version('matplotlib')}")


pandas version: 2.3.3
scanpy version: 1.11.5
matplotlib version: 3.10.8


In [3]:
from module.config_local import sample_name_import

# name_dir = "circa-SD"
# name_dir = 'all-samples-C123'
# name_dir = 'all-samples-C0'
name_dir = "all-samples-combined"
# name_dir = "liver-cancer"

samples_ids = sample_name_import(name_dir)

print(len(samples_ids))
print(samples_ids)

12
['2505-1', '2505-2', '2670-1', '3159-1', '3160-1', '3160-2', '3159-2', '3161-1', '3159-3', '3161-2', '3159-4', '3161-3']


# Data import

In [4]:
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")

# adata = sc.read_h5ad(f"{dir_processed}/h5ad/AD_unassigned/AD_{name_dir}_unassigned_adata.h5ad")

In [ ]:
if 'leiden_colors' in adata.obs:
    adata.obs = adata.obs.drop(columns=['leiden_colors'])

if not os.path.exists(f"{dir_processed}/h5ad/{name_dir}/"):
   os.makedirs(f"{dir_processed}/h5ad/{name_dir}/")

adata.write_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")

In [ ]:
df = pd.read_parquet(f'{dir_processed}/csv/{name_dir}/{name_dir}_norm_combined.parquet')

In [ ]:
df = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
df.shape

from module.xenium_preprocessing import add_annotations

df = add_annotations(adata, df)

In [ ]:
df.to_parquet(f'{dir_processed}/csv/{name_dir}/{name_dir}_norm_combined.parquet')

In [ ]:
ordered_categories = adata.uns['dendrogram_']['categories_ordered']
print(*ordered_categories, sep='","')

# Visualization

## UMAP

In [ ]:
adata.obs['cell_type_final'].unique()

In [ ]:
da.umap_plot_indi_multi(adata,
                     name_dir,
                     cluster_to_use = "cell_type_newnum_final", ### Use the column with numbers, not with actual name
                     individual_plot = False,
                     save_plot = True,
                     cmap_ = 'tab20b',
                     )


In [ ]:
adata.obs['umap-1'] = adata.obsm['X_umap'][:, 0]
adata.obs['umap-2'] = adata.obsm['X_umap'][:, 1]

In [ ]:
### UMAP plot with gene expression as color scale
 
sc.pl.umap(adata, color="Genotype", vmin = 0)

## Scatter plot

In [ ]:
da.cluster_plot(adata,
             name_dir,
             cluster_to_use = 'cell_type_newnum_final',
             cluster_to_map = ['all'],
             cmap_ = 'tab20b',
             save_plot = True,
            )

## Polygon plots

### Data prep

#####################################################################################################################
First process cells polygons were extracted and saved as GEOjson [HERE](./v11_Polygons_preprocessing.ipynb)
#####################################################################################################################

In [ ]:

cluster_to_use = "leiden"

sample_to_plot = "hLiver-cancer"

df, cells_geo, cluster_to_use = da.polygonplot_dataprep(adata,
                                    sample_to_plot,
                                    cluster_to_use = cluster_to_use,
                                    cmap_ = 'tab20b',
                                    )

d:\Jupyter_notebook\Xenium_jupyter_notebook\Xenium_workflow\Xenium-workflow\module\dataviz_analysis.py:279: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cells_geo['centroid'] = cells_geo['geometry'].centroid


KeyError: 'region_automap_name'

In [ ]:
df.head(2)

In [ ]:
cells_geo.sort_values(by='cell type', inplace=True)
cells_geo.head(5)

### Celltype + genes as symbols

In [ ]:
from module.dataviz_analysis import polygonplot_plot

polygonplot_plot(df, cells_geo, name_dir,
                cluster_to_use = "cell type",
                gene_ = None, # example = 'Gfap'
                # region_only = "CTX", # example : 'SCH' ### Priority 1
                region_ = 'SCH', # example : 'SCH' ### Priority 2
                # coord_ = None, # example : [1000,2000,2000,3000] ### Priority 3
                # coord_ = [5700,6900,3600,4900], ### Priority 3
                legend = True,
                save_plot = False)


### Gene expression as color gradient

In [ ]:
from module.dataviz_analysis import polygonplot_plot_gradient

polygonplot_plot_gradient(df, cells_geo,
                          gene_ = 'Gfap', ## Required
                          region_ = 'HIPP',
                          region_only = None,
                          coord_ = [2000,4000,2000,4000],
                          cmap_ = 'bwr',
                          save_plot = False)


## Other plots

In [ ]:
### Plot gene expression


# sns.set_theme(style="white")

df = pd.read_parquet(f'{dir_processed}/csv/{name_dir}/{name_dir}_norm_combined.parquet')

gene_ = 'Chat'
adata_temp = adata
df_dict = dict(zip(df.index, df[gene_]))
adata_temp.obs[gene_] = adata_temp.obs['cell_id'].map(df_dict)

### to crop
adata_temp = adata_temp[(adata_temp.obs['x_centroid'] > 4000) & (adata_temp.obs['x_centroid'] < 6000)
                            & (adata_temp.obs['y_centroid'] > 4000) & (adata_temp.obs['y_centroid'] < 5000)]

# f, ax = plt.subplots(figsize=(15, 10))

fig, axs = plt.subplots(3,2#,figsize=(15, 15)
                        )
axs = axs.flatten()# Mapping of clusters

for idx, sample in enumerate(samples_ids):
    adata_graph = adata_temp[adata_temp.obs['sample'] == sample]
    
    sns.scatterplot(x='x_centroid', y='y_centroid',
                s=2, legend= False,
                palette='viridis',
                hue=gene_,
                data=adata_graph.obs, ax=axs[idx]).set(title=f"{sample} - {gene_}", xlabel = None, ylabel = None, xticklabels = [],yticklabels = [])

# Create the colorbar
norm = mpl.colors.Normalize(vmin=adata_temp.obs[gene_].min(),vmax=adata_temp.obs[gene_].max())
sm = mpl.cm.ScalarMappable(cmap='viridis', norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axs[-1], aspect=40, shrink=0.8)  # Adjust aspect/shrink

del adata_temp, adata_graph


### Gene expression accross samples

In [ ]:
### Gene expression accross samples
if 'df' not in globals():
    df = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
    df['cell_id'] = df.index

dict_ = dict(zip(adata.obs['cell_id'],adata.obs['sample']))
df['sample'] = df['cell_id'].map(dict_)

gene = 'Arc'
x = df.groupby('sample')[gene].mean()

plt.scatter(x = x.index, y=x)
plt.tick_params(rotation=90)
plt.ylim(0,)
plt.title(f'Ndn expression of {gene}')
plt.ylabel('Normalized expression')

In [ ]:
# to_use = 'total_transcript' ### Resegmented only
to_use = 'transcript_counts' ### 

### Map log transcript counts 
adata_sel = adata[(adata.obs['sample'] == samples_ids[0])]
adata_sel.obs[to_use] = adata_sel.obs[to_use].astype(float)
adata_sel.obs['log_transcript_counts'] = adata_sel.obs[to_use].apply(lambda x: math.log10(x))

fig, ax = plt.subplots(figsize=(10,6))
transcript_counts_unique = adata_sel.obs['log_transcript_counts'].unique()
cmap = plt.cm.jet
for cluster_id in transcript_counts_unique:
    cluster_data = adata_sel.obs[adata_sel.obs['log_transcript_counts'] == cluster_id]
    colors = cmap((cluster_id - transcript_counts_unique.min()) / (transcript_counts_unique.max() - transcript_counts_unique.min()))
    plt.scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=0.5, label=cluster_id)
plt.xlabel('x_centroid')
plt.ylabel('y_centroid')
plt.title('Map of transcript counts (log)')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap)
sm.set_array(transcript_counts_unique)
divider = make_axes_locatable(ax)
cbar_ax = divider.append_axes('right', size='1.5%', pad=0.05)
plt.colorbar(sm, cax=cbar_ax, label='Transcript Counts')
plt.ylabel('Transcript counts (log)')

# plt.savefig(f"{dir_processed}/plot/{name_dir}/{name_dir}_logtranscounts.png")

In [ ]:
# to_use = 'n_genes' ### Resegmented only
to_use = 'n_genes_by_counts' ### 

### Map log n_genes_by_counts 
adata_sel = adata[(adata.obs['sample'] == samples_ids[0])]
adata_sel.obs[to_use] = adata_sel.obs[to_use].astype(float)
adata_sel.obs['log_n_genes_by_counts'] = adata_sel.obs[to_use].apply(lambda x: math.log10(x))

fig, ax = plt.subplots(figsize=(10,6))
transcript_counts_unique = adata_sel.obs['n_genes_by_counts'].unique()
cmap = plt.cm.jet
for cluster_id in transcript_counts_unique:
    cluster_data = adata_sel.obs[adata_sel.obs['n_genes_by_counts'] == cluster_id]
    colors = cmap((cluster_id - transcript_counts_unique.min()) / (transcript_counts_unique.max() - transcript_counts_unique.min()))
    plt.scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=0.5, label=cluster_id)
plt.xlabel('x_centroid')
plt.ylabel('y_centroid')
plt.title('Map of Nb of gene per cell')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap)
sm.set_array(transcript_counts_unique)
divider = make_axes_locatable(ax)
cbar_ax = divider.append_axes('right', size='1.5%', pad=0.05)
plt.colorbar(sm, cax=cbar_ax)
plt.ylabel('Nb of gene per cell')

# plt.savefig(f"{dir_processed}/plot/{name_dir}/{name_dir}_nbgenes.png")

In [ ]:
### Map mmc:cluster_correlation_coefficient
adata_sel = adata[(adata.obs['sample'] == samples_ids[0])]

fig, ax = plt.subplots(figsize=(10,6))
transcript_counts_unique = adata_sel.obs['mmc:class_correlation_coefficient'].unique()
cmap = plt.cm.jet
for cluster_id in transcript_counts_unique:
    cluster_data = adata_sel.obs[adata_sel.obs['mmc:class_correlation_coefficient'] == cluster_id]
    colors = cmap((cluster_id - transcript_counts_unique.min()) / (transcript_counts_unique.max() - transcript_counts_unique.min()))
    plt.scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=0.5, label=cluster_id)
plt.xlabel('x_centroid')
plt.ylabel('y_centroid')
plt.title('Map of mmc:class_correlation_coefficient')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap)
sm.set_array(transcript_counts_unique)
divider = make_axes_locatable(ax)
cbar_ax = divider.append_axes('right', size='5%', pad=0.05)
plt.colorbar(sm, cax=cbar_ax, label='correlation_coefficient')

# plt.savefig(f"{dir_processed}/plot/{name_dir}/{name_dir}_mmccoef.png")

## Gene expression plots

### Plot individual genes

In [ ]:
df = pd.read_parquet(f'{dir_processed}/csv/{name_dir}/{name_dir}_norm_combined.parquet')

In [ ]:
expe = "Genotype"
control_grp = "WT"
experimental_grp = "APP"

df_ctrl = df[(df[expe]==control_grp)]
df_expe = df[(df[expe]==experimental_grp)]

In [ ]:
gene_name = 'Vip'

plt.plot(df_ctrl.groupby('sample')[gene_name].mean(), "bo-", label = 'Ctrl' )
plt.plot(df_expe.groupby('sample')[gene_name].mean(), "go--", label = 'Expe')
plt.title(gene_name)
plt.legend()

In [ ]:
plt.figure(figsize=(3,5))
plt.bar(x = control_grp, height = df_ctrl[gene_name].mean(), color = '#EE2A35')
plt.errorbar(x=control_grp,  y = df_ctrl[gene_name].mean(), yerr=df_ctrl[gene_name].std()/ math.sqrt(df_ctrl.shape[0]), color = 'black')
plt.bar(x = experimental_grp, height = df_expe[gene_name].mean(), color = '#009736')
plt.errorbar(x=experimental_grp,  y = df_expe[gene_name].mean(), yerr=df_expe[gene_name].std()/ math.sqrt(df_expe.shape[0]), color = 'black')
plt.ylim(0)
plt.title(gene_name)


### Define marker genes

In [ ]:
marker_genes = [
# 10X annotations
# "Acsbg1","Aqp4","Cdh20","Clmn","Gfap","Gli3","Id2","Mapk4","Ntsr2","Pde7b","Rfx4","Rorb","Slc39a12", #Astrocytes
# "Arhgap12","Fibcd1","Sipa1l3","Wfs1", #CA1-ProS
# "2010300C02Rik","Arhgef28","Bcl11b","Bhlhe22","Cabp7","Cpne4","Igfbp4","Necab2","Prdm8","Strip2","Syndig1", #CA2
# "Cpne6","Epha4","Hat1","Neurod6","Npy2r","Nrp2","Shisa6", #CA3
# "Cdh9","Orai2","Prox1","Rasl10a","Tanc1", #DG
# "Acvrl1","Adgrl4","Car4","Cd93","Cldn5","Cobll1","Emcn","Fgd5","Fn1","Kdr","Ly6a","Mecom","Nostrin","Paqr5","Pecam1","Pglyrp1","Slfn5","Sox17","Zfp366", #Endothelial
# "Arhgap25","Cd300c2","Cd53","Cd68","Ikzf1","Laptm5","Siglech","Sla","Spi1","Trem2",#Microglia
# 'Gjc3','Gpr17','Opalin','Sema3d','Sema6a','Sox10','Zfp536', #Oligodendrocytes
# 'Acta2','Ano1','Arhgap6','Carmn','Cspg4','Fos','Gucy1a1','Inpp4b','Nr2f2','Pip5k1b','Plekha2','Pln','Sncg','Sntb1', # Pericytes
# 'Aldh1a2','Col1a1','Col6a1','Cyp1b1','Dcn','Fmod','Gjb2','Igf2','Pdgfra','Ror1',"Slc13a4","Spp1", #VLMC
# "Chat","Crh","Igf1",'Penk','Pthlh','Sorcs3','Thsd7a','Vip', #Vip interneurons

# 'Arntl','Clock','Cry1','Cry2','Nr1d1',"Per1",'Per2','Per3','Rora','Rorb', ## Clock genes
# 'Gfap','Trem2','Cd44','Spp1','Cd68','Igf1','Spi1','Cd300c2','Cd53','Laptm5','Ikzf1','Arhgap25','Opalin','Prox1','Cbln1','Sema3a','Paqr5','Spag16',
# "Vip", "Pkib", "Tmem255a", "Arhgap6", "Chodl", #SCN rank genes analysis
'Strip2',"Shisa6","Chodl", 'Fos','Sdk2', 'Cdh6','Cobll1','Tanc1'
# 'Dner','Gad1','Rasgrf2','Vat1l','Pde7b','Igfbp5','Rorb','Rims3','Tmem255a','Cdh13','Gad2','Rab3b','Parm1','Tle4','Fhod3','Rmst','Vip','Nr2f2','Arhgap6',
# 'Laptm5','Kctd12','Siglech','Trem2','Cd53','Cd68','Cd300c2','Ikzf1','Spi1','Acsbg1','Gfap','Dpy19l1','Unc13c','Arhgap25','Meis2','Dner','Arhgap12','Igfbp5','Ntsr2',
# "Gfap","Rbp4","Trem2","Th","Laptm5","Syt17","Opn3","Spp1","Cd44","Cd53","Igf1","Gjb2",
# "Vip","Prss35","Cd68","Cplx3","Siglech","Ikzf1","Cd300c2","Dcn","Spi1","Pkib","Fos","Angpt1",
# "Igfbp5","Chrm2","Rspo2","Arhgap25","Sst",
# 'Ntsr2'    
]


### Stacked violin plots

Should probably be run only on subset of adata corresponding to a cell type or a region.

In [ ]:
ax = sc.pl.stacked_violin(adata, ### Can be more useful on subset of the data, otherwise "zero values" greatly change the graph
                         marker_genes, ### marker_genes or individual genes (ex: "Dner")
                         groupby='Genotype',
                         dendrogram=False,
                         log=True,
                         )

### Violin plot for individual genes with individual data point (1 graph/gene)

In [ ]:
sc.pl.violin(adata, marker_genes, groupby='Genotype', order = ['WT','APP'],
             jitter = 0.45,
             log = True,
             stripplot = False,
            )

### Heatmap

In [ ]:
from module.misc import genes_list

# marker_genes = genes_list('brain_panel')
# marker_genes = genes_list('panel_5k_circa')
# marker_genes = adata.var.index
marker_genes = genes_list('mitochondria')

adata_sub = adata[:, adata.var_names.isin(marker_genes)].copy()

df = pd.DataFrame(data=adata_sub.X.toarray(), index=adata_sub.obs_names, columns=adata_sub.var_names)


In [ ]:
# import pandas as pd
# import scanpy as sc

# df = pd.read_parquet('D:\\Jupyter_notebook\Xenium_jupyter_notebook\csv/circa-SD/circa-SD_norm_combined.parquet', columns = clock_genes)

In [ ]:
from module.xenium_preprocessing import add_annotations

df = add_annotations(adata_sub, df)

In [ ]:
df.shape

In [ ]:
marker_genes = adata_sub.var.index

In [ ]:
df

In [ ]:
sc.pl.heatmap(adata_sub, var_names = marker_genes, groupby=["ZT"],
               standard_scale='obs', dendrogram=False, swap_axes=True)

In [ ]:
grouped = df.groupby(["cell_type_final"])[marker_genes].mean()
grouped.shape


In [ ]:
# ordered_categories =['SMC', 'Sncg Gaba', 'STR Gaba', 'COAa PAA MEA Glut', 'OB STR CTX IMN', 'MEA Glut', 'L5 NP CTX Glut', 'PAL STR Gaba Chol',
#  'Vip Gaba', 'Lamp5 Gaba', 'CLA EPd CTX Glut', 'STR PAL Gaba', 'L6b CTX Glut', 'Sst Gaba', 'VLMC', 'PVH Glut', 'ABC', 'AHN Glut', 'SCH Gaba', 'L5 ET CTX Glut', 'Choroid', 'Pericyte', 'LHA Glut', 'Pvalb Gaba',
#  'Ependymal',  'OPC', 'Microglia', 'L2 3 IT PIR ENTl Glut', 'L6 CT CTX Glut', 'Endothelial', 'STR D1 Gaba', 'L4 5 IT CTX Glut', 'Oligodendrocyte',
#  'STR D2 Gaba', 'L6 IT CTX Glut', 'Astro TE']

# ordered_categories = ['Oligodendrocyte', 'OPC', 'Astro NT', 'Astro TE', 'Ependymal', 'Pineal Glut', 'Tanycyte', 'CHOR', 'VLMC', 'Endothelial', 'Pericyte', 'Microglia', 'CA1 ProS Glut', 'CA2 FC IG Glut', 'CA3 Glut', 'DG Glut', 'L23 PIR ENTl Glut', 'MEA Glut', 'LA Glut', 'NLOT Glut', 'L23 CTX Glut', 'L23 RSP Glut', 'L4 CTX Glut', 'L6 CTX Glut', 'L5 CTX Glut', 'SUB ProS Glut', 'L6b CTX Glut', 'AD Glut', 'AV Glut', 'TH Glut', 'SN Dopa', 'LHA Glut', 'MB Glut', 'PAG Glut', 'HY Glut', 'LH Glut', 'VMH Glut', 'MM Glut', 'PVT Glut', 'PF Glut', 'APN Glut', 'SC Glut', 'MH Glut', 'BST Glut', 'LSX Gaba', 'SCH Gaba', 'Sst Gaba', 'MEA Gaba', 'BST Gaba', 'HY GABA', 'ARH GABA', 'Lamp5 Gaba', 'Vip Gaba', 'STR Gaba', 'STRv PAL Gaba', 'LGv Gaba', 'PRT Gaba', 'SC Gaba', 'ZI Gaba', 'SN Gaba', 'RT ZI GABA', 'Pvalb Gaba', 'STR D1D2 Gaba']
ordered_categories = ['Oligodendrocyte', 'Microglia', 'Choroid', 'ABC', 'VLMC', 'OB_STR_CTX_Inh_IMN', 'Ependymal', 'Tanycyte', 'OPC', 'Astrocyte', 'Endothelial', 'Pericyte', 'PVH_SO_Glut', 'PVR_Gaba', 'SCH_Gaba', 'MH_Glut', 'TRS_BAC_Glut', 'DG_PIR_Ex_IMN', 'HPF_CR_Glut', 'LSX_Gaba', 'CEA_AAA_BST_Gaba', 'STR_PAL_Gaba', 'Sst_Gaba', 'Lamp5_Gaba', 'Vip_Gaba', 'Pvalb_Gaba', 'STR_Gaba', 'RT_ZI_Gaba', 'SI_Gaba', 'ATN_Glut', 'PT_Glut', 'RE_Glut', 'AD_Glut', 'AV_Glut', 'TH_Glut', 'LSX_Glut', 'BST_MPN_Gaba', 'COAa_PAA_MEA_Glut', 'PAL_STR_Gaba_Chol', 'PVT_Glut', 'BST_po_Glut', 'LH_Glut', 'MEA_Glut', 'AHN_Glut', 'LHA_Glut', 'STR_D1_Gaba', 'STR_D2_Gaba', 'DG_Glut', 'CA1_ProS_Glut', 'CA2_FC_IG_Glut', 'CA3_Glut', 'L2_3_IT_CTX_Glut', 'L4_5_IT_CTX_Glut', 'L4_RSP_ACA_Glut', 'L2_3_IT_RSP_Glut', 'L5_ET_CTX_Glut', 'L5_NP_CTX_Glut', 'L6_IT_CTX_Glut', 'L6_CT_CTX_Glut', 'L6b_CTX_Glut', 'CLA_EPd_CTX_Glut', 'L2_3_IT_PIR_ENTl_Glut', 'NLOT_Glut']


In [ ]:

grouped_sort = grouped.reindex(ordered_categories)
grouped_sort.dropna(axis=0, inplace=True)
grouped_sort = grouped_sort.reindex(sorted(grouped_sort.columns), axis=1)
# grouped_sort.drop('Tanycyte',axis=0,inplace = True)

In [ ]:
gene_list = grouped_sort.columns
cell_types = grouped_sort.index

In [ ]:
sns.set_theme(font_scale=0.5)
sns.clustermap(grouped_sort, cmap = 'bwr', z_score=1,  center=0, vmin = -3, vmax = 3,
                col_cluster=True,row_cluster=True, cbar = 'upper left', cbar_pos=(-0.05,0.1,0.1,0.5),figsize=(15, 10),
                )
plt.ylabel(ylabel=None)
# plt.savefig(f'Gallery/{today}/heatmap_clockgenes_allcelltypes.svg',format = "svg", transparent = True, dpi=300)

#### Heatmap per Zt

In [ ]:
import pickle

with open(f"{dir_processed}/analysis/circa-SD/cycling_genes_database.pickle", "rb") as handle:
    dict_all_cycling = pickle.load(handle)

key = 'SCH_Gaba'

In [ ]:
CC_genes = dict_all_cycling[key]['CircaCompare'][dict_all_cycling[key]['CircaCompare']["P-value for difference in phase"]<0.05].sort_values(by="Phase difference estimate")[['gene','circa4 peak time hours']][0:40]
# CC_genes = dict_all_cycling[key]['CircaCompare'][dict_all_cycling[key]['CircaCompare']["P-value for difference in phase"]>0.05].sort_values(by="Phase difference estimate")[['gene','circa4 peak time hours']]#[0:40]


In [ ]:
CC_genes_list = CC_genes.sort_values(by='circa4 peak time hours')['gene'].values
CC_genes_list

In [ ]:
dict_all_cycling[key].keys()

In [ ]:
NS_phase = dict_all_cycling[key]["NS_cyc"].sort_values(by="meta2d_pvalue")[["CycID", "meta2d_phase", "meta2d_pvalue"]]#[0:40]
SF_phase = dict_all_cycling[key]["SD_cyc"].sort_values(by="meta2d_pvalue")[["CycID", "meta2d_phase", "meta2d_pvalue"]]#[0:40]

NS_phase_sig = dict_all_cycling[key]["NS_cyc_filter"].sort_values(by="meta2d_pvalue")[["CycID", "meta2d_phase", "meta2d_pvalue"]]
SF_phase_sig = dict_all_cycling[key]["SD_cyc_filter"].sort_values(by="meta2d_pvalue")[["CycID", "meta2d_phase", "meta2d_pvalue"]]

NS_phase_unique = NS_phase_sig[~NS_phase_sig.index.isin(SF_phase_sig.index)]
SF_phase_unique = SF_phase_sig[~SF_phase_sig.index.isin(NS_phase_sig.index)]

NS_cyc_genlist = NS_phase_unique.sort_values('meta2d_phase')['CycID'].values
SF_cyc_genlist = SF_phase_unique.sort_values('meta2d_phase')['CycID'].values

In [ ]:
len(NS_phase_unique),len(SF_phase_unique)

In [ ]:
NS_cyc_genlist = CC_genes_list
# NS_cyc_genlist = NS_cyc_genlist#[0:40]
# NS_cyc_genlist = SF_cyc_genlist[50:90]

In [ ]:
NS_cyc_genlist

In [ ]:
# adata_sub = adata[:, adata.var_names.isin(clock_genes)].copy()
# adata_sub = adata[:, adata.var_names.isin(marker_genes)].copy()

df_NS = pd.DataFrame(data=adata[:, adata.var_names.isin(NS_cyc_genlist)].X.toarray(),
                     index=adata[:, adata.var_names.isin(NS_cyc_genlist)].obs_names,
                     columns=adata[:, adata.var_names.isin(NS_cyc_genlist)].var_names)

from module.xenium_preprocessing import add_annotations

df_NS = add_annotations(adata, df_NS)


In [ ]:
grouped = df_NS[(df_NS['cell_type_final']=='SCH Gaba') & (df_NS['run']=='circa4')].groupby(["ZT"])[NS_cyc_genlist].mean()
grouped.shape

grouped_SF = df_NS[(df_NS['cell_type_final']=='SCH Gaba') & (df_NS['run']=='SD1')].groupby(["ZT"])[NS_cyc_genlist].mean()
grouped_SF.shape

In [ ]:
column_sums = grouped.sum(axis=0)
columns_to_keep = column_sums[column_sums != 0].index
grouped_filtered = grouped[columns_to_keep]

column_sums_SF = grouped_SF.sum(axis=0)
columns_to_keep_SF = column_sums_SF[column_sums_SF != 0].index
grouped_filtered_SF = grouped_SF[columns_to_keep_SF]

In [ ]:
markers = ['Vip', "Marcks", "Cadps2", "Fam49b", "Ppp3r1", "Peg10", "Per1", "Odc1", "Nr1d1", "Dbp"]
grouped_filtered = df.groupby('sample')[markers].mean()

In [ ]:
sns.clustermap(grouped_filtered.T, cmap = 'Blues', z_score=1,  center=0,# vmin = -2, vmax=2,
                col_cluster=False,row_cluster=False, cbar = 'upper left', cbar_pos=(1.0,0.1,0.1,0.7),
                figsize=(5,7),
                )
plt.ylabel(ylabel=None)
plt.title('NS_CC_diff_phase')
# plt.savefig(f'Gallery/{today}/heatmap_SCH_NS_CC_diff_phase_all.svg',format = "svg", transparent = True, dpi=300)

In [ ]:
sns.clustermap(grouped_filtered_SF.T, cmap = 'Blues', z_score=0,  center=0,# vmin = -2, vmax=2,
                col_cluster=False,row_cluster=False, cbar = 'upper left', cbar_pos=(1.0,0.1,0.1,0.7),
                figsize=(3,7),
                )
plt.ylabel(ylabel=None)
plt.title('SF_CC_diff_phase')
plt.savefig(f'Gallery/{today}/heatmap_SCH_SF_CC_diff_phase_all.svg',format = "svg", transparent = True, dpi=300)

# Analysis

## Highest expressed genes

In [ ]:
# adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_automap.h5ad.gz")
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")

# if 'leiden_colors' in adata.obs:
#     adata.obs = adata.obs.drop(columns=['leiden_colors'])
# adata.write_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_automap.h5ad.gz")

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)

In [ ]:
# adata_WT=adata[adata.obs['Genotype']== 'WT']
# adata_APP=adata[adata.obs['Genotype']== 'APP']
grp = adata.obs['run'].unique()

sc.pl.highest_expr_genes(adata[adata.obs['run'] == grp[0]], n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)
sc.pl.highest_expr_genes(adata[adata.obs['run'] == grp[1]], n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)

## Find marker genes for each cluster

In [ ]:
adata.obs['cell_type_final'] = pd.Categorical(adata.obs['cell_type_final'])
adata.obs['cell_type_final'].cat.categories


In [ ]:
sc.tl.dendrogram(adata,groupby='cell_type_final', key_added='dendrogram_')

In [ ]:
# Obtain cluster-specific differentially expressed genes

# cluster_to_use = 'cell_type_newnum'
# cluster_to_use = 'cell type'
# cluster_to_use = 'cell_type_auto'
# cluster_to_use = 'cell_type_auto_sub'
cluster_to_use = 'cell_type_final'
# cluster_to_use = 'Genotype'

adata.obs[cluster_to_use] = adata.obs[cluster_to_use].astype(str)
sc.tl.rank_genes_groups(adata, groupby=cluster_to_use, method="wilcoxon", tie_correct = True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata,
                                n_genes=5,
                                values_to_plot="logfoldchanges",
                                cmap='bwr',
                                # vmin=-4,
                                # vmax=4,
                                )

### Hierarchical clustering

In [ ]:
adata.obs['cell_type_final'].nunique()

In [ ]:
adata.uns.keys()

In [ ]:
ordered_categories = adata.uns['dendrogram_cell_type_final']['categories_ordered']
print(ordered_categories)

In [ ]:
dict_update = dict(zip(ordered_categories, range(83)))
adata.obs['cell_type_newnum_final'] = adata.obs['cell_type_final'].map(dict_update)

In [ ]:
adata.obs['celltype_number'] = adata.obs['cell_type_newnum_final'].astype(str) + '_' + adata.obs['cell_type_final'].astype(str)

In [ ]:
adata.obs['celltype_number'] = adata.obs['celltype_number'].astype('category')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby=cluster_to_use,
                                values_to_plot="logfoldchanges",
                                min_logfoldchange= 0.26,
                                standard_scale="var",
                                n_genes=2, dendrogram = True)

### Extract all cluster compared to all others in a single sheet


In [ ]:
### Extract all cluster compared to all others in a single sheet

dat = pd.DataFrame()
for i in adata.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata, group=str(i))
    dat1['group'] = i
    dat = pd.concat([dat, dat1])

dat.to_csv(f"{dir_processed}/analysis/{name_dir}/{name_dir}_{cluster_to_use}_markergenes.csv")

In [ ]:
### Compare two groups gene expression (whole section)
section_ = 'C3'
adata_temp = adata[adata.obs['section']==section_]
sc.tl.rank_genes_groups(adata_temp, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)
dat1 = sc.get.rank_genes_groups_df(adata_temp, group='APP')
dat1['group'] = 'APP'

dat1 = dat1[ ### Choose filters here
# (dat1['pct_nz_group'] > 0.15) & #Percentage of cell expressing the gene
(dat1['pvals_adj']<= 0.05) & # adjusted p-value
(abs(dat1['logfoldchanges']) > 0.26) # logfoldchange
]

dat1.to_csv(f"csv/{name_dir}/foldchange/{name_dir}_{section_}_whole_section_filter.csv")

## Subset (a)dataset for one cell type

In [ ]:
adata.obs['cell_type_final'].value_counts()

In [ ]:
adata2 = adata
# del adata
celltype_to_subset = "SCH_Gaba"
# region_to_subset = "SCH"

condition_to_use = "Genotype" ## AD
order_to_use = ['WT','APP']

# condition_to_use = "run" ## SD
# order_to_use = ['circa4', "SD1"]

adata_microglia = adata2[
                        (adata2.obs['cell_type_final'] == celltype_to_subset)
                        # & (adata2.obs['region_automap_name'] == region_to_subset)
                         ]

In [ ]:
samples_ids_sub = adata_microglia.obs['sample'].unique()

In [ ]:
marker_genes = ["Gfap","Camk2a"]
# , "Spp1", "Fos","Trem2","Cd44","Laptm5","Ebf3","Cpne8","Igf1","Spi1",
#                 "Prss35","Cd300c2","Cd53","Cd68","Prph","Dcn","Igfbp5","Slc6a3",	"Angpt1", "Cbln1", "Ccn2"]

In [ ]:
sc.pl.stacked_violin(adata_microglia, marker_genes, groupby=condition_to_use, dendrogram=False,)

In [ ]:
sc.pl.violin(adata_microglia, marker_genes, groupby=condition_to_use, order = order_to_use,
             log = True,
            #  stripplot = False,
            )

In [ ]:
adata_microglia.obs['Genotype'].unique()

In [ ]:
sc.tl.rank_genes_groups(adata_microglia, groupby=condition_to_use, method="wilcoxon", tie_correct = True, corr_method="benjamini-hochberg", pts = True)
dat1 = sc.get.rank_genes_groups_df(adata_microglia, group = order_to_use[1])

# dat1.to_csv(f'{dir_processed}/analysis/{name_dir}/{celltype_to_subset}_fold_change_no_threshold.csv', index=False)

In [ ]:
dat1 =  dat1.sort_values(by='logfoldchanges', ascending=False)
# dat1 = dat1[(dat1['pct_nz_group'] > 0.15)]
dat1#[dat1['names']=="Lhx1"]

In [ ]:
dat1_filter = dat1[(~dat1['logfoldchanges'].between(-0.26,0.26))
                   & (dat1['pvals_adj'] <= 0.05)
                    & (dat1['pct_nz_group'] > 0.05)
                      ]
dat1_filter.reset_index(inplace=True)
dat1_filter.to_csv(f'{dir_processed}/analysis/{name_dir}/{celltype_to_subset}_fold_change.csv', index=False)

### Volcano plot

In [ ]:
dat1_filter.index = dat1_filter.names

In [ ]:
dat1

In [ ]:
# dat1_filter = dat1[(~dat1['logfoldchanges'].between(-0.26,0.26)) & (dat1['pvals_adj'] <= 0.05) ]
# dat1_filter = dat1_filter.reset_index
# create sub pandas with genes over threshold
# plot after initial scatter with alpha = 1
# add text for each postion

dat1['Q'] = -np.log10(dat1['pvals_adj'])

dat1_filter['Q'] = -np.log10(dat1_filter['pvals_adj'])

plt.vlines(x=(-0.26,0.26), ymin=0, ymax=250, color = "black", linestyles='dashed')
plt.hlines(y=0.05, xmin=-1, xmax=1, color = "black", linestyles='dashed')
plt.scatter(x=dat1['logfoldchanges'], y = dat1['Q'], alpha= 0.75, color = "grey", edgecolors=None,s=2)
plt.scatter(x= dat1_filter['logfoldchanges'], y=dat1_filter['Q'], alpha=1, color = 'red',s=2)
# 
#     

texts = []
for gene in dat1_filter['names'].unique():
    if dat1_filter.loc[gene,'Q'] <= 50: continue
    texts.append(plt.text(dat1_filter.loc[gene,'logfoldchanges'], dat1_filter.loc[gene, 'Q'], str(gene), color = 'black', fontsize = 6, ha= 'center'))
# adjust_text(texts,avoid_self= True, prevent_crossings= True, arrowprops=dict(arrowstyle="-", color='black', lw=0.5))

# plt.yscale('log')
# plt.xlim(-4,4)
# plt.gca().invert_yaxis()
plt.xlabel('Log2 Foldchange')
plt.ylabel('-log10(ajusted p-value)')
plt.savefig(f'Gallery/{today}/{celltype_to_subset}_{name_dir}_volcano_plot.svg')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_microglia,
                                n_genes=10,
                                values_to_plot="logfoldchanges", cmap='bwr',
                                # vmin=-4,
                                # vmax=4,
                                )

In [ ]:
sc.pl.highest_expr_genes(adata_microglia, n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)

In [ ]:
adata_microglia_WT=adata_microglia[adata_microglia.obs['Genotype']== 'WT']
adata_microglia_APP=adata_microglia[adata_microglia.obs['Genotype']== 'APP']

sc.pl.highest_expr_genes(adata_microglia_WT, n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)
sc.pl.highest_expr_genes(adata_microglia_APP, n_top=30, show=None, save=None, ax=None, gene_symbols=None, log=False)

In [ ]:
adata.obs[(adata.obs['cell_type_final_bis']=="Astrocyte")].groupby(['region_automap_name','Genotype'])['cell_type_final'].count()

### Subcluster the subset

In [ ]:
adata_microglia

In [ ]:
# extract pca coordinates


sc.pp.pca(adata_microglia, n_comps = 50)
sc.pp.neighbors(adata_microglia)
sc.tl.umap(adata_microglia, min_dist = 1)

X_pca = adata_microglia.obsm['X_pca'] 

### Kmeans clustering
### You can choose the number of clusters by uncommenting n_clusters option
# kmeans = KMeans(n_clusters=2,
#                 random_state=0).fit(X_pca) 
# adata_microglia.obs['kmeans'] = kmeans.labels_.astype(str)

sc.tl.leiden(adata_microglia, resolution = 0.2, flavor = "leidenalg", n_iterations=2)

In [ ]:
### Choose one cluster to work with

# cluster_to_use = 'kmeans'
cluster_to_use = 'leiden'


In [ ]:
### Number of cells per clusters

table_count = adata_microglia.obs.groupby(condition_to_use)[cluster_to_use].value_counts(normalize=True).sort_index()
table_count
ticks = table_count.index
ticks = [tick[1] for tick in ticks]
ticks

plt.figure(figsize=(2,3))
plt.bar(x=range(len(ticks)), height=table_count.values)
plt.xticks(ticks = range(len(ticks)), labels = ticks)
plt.ylabel(f'Proportion of subclusters')
plt.title(celltype_to_subset)
plt.savefig(f'Gallery/{today}/{celltype_to_subset}_R1_barplot.svg', dpi= 300, format='svg', transparent = True)

In [ ]:
### Generate a color palette for the clusters - to make color stay consistent across samples
adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)

# Create a palette with a unique color for each cluster
num_clusters = len(adata_microglia.obs[cluster_to_use].astype(int).unique())
palette = sns.color_palette("hls", n_colors=num_clusters)

# Map each 'leiden' value to a color
adata_microglia.obs['leiden_colors'] = adata_microglia.obs[cluster_to_use].astype(int).apply(lambda x: palette[x])

In [ ]:
### Let's make UMAP plot. We will also add the cluster centroids to the plot
adata_microglia.obs['umap-1'] = adata_microglia.obsm['X_umap'][:, 0]
adata_microglia.obs['umap-2'] = adata_microglia.obsm['X_umap'][:, 1]
cluster_centroids = adata_microglia.obs.groupby(cluster_to_use)[['umap-1', 'umap-2']].median()

In [ ]:
from module.dataviz_analysis import umap_plot_indi_multi
samples_ids = samples_ids

umap_plot_indi_multi(adata_microglia[adata_microglia.obs['Genotype']=="WT"],
                     name_dir = name_dir,
                     dir_processed=dir_processed,
                     cluster_to_use = "leiden",
                     individual_plot = False,
                     save_plot = False,
                     cmap_ = 'hls',
                     )

In [ ]:
from matplotlib.pyplot import rc_context
with rc_context({"figure.figsize": (5, 5)}):
    sc.pl.umap(
        adata_microglia[adata_microglia.obs['Genotype']=="WT"],
        color=cluster_to_use,
        add_outline=False,
        legend_loc="on data",
        legend_fontsize=12,
        legend_fontoutline=2,
        frameon=True,
        palette="tab20",
    )

with rc_context({"figure.figsize": (5, 5)}):
    sc.pl.umap(
        adata_microglia[adata_microglia.obs['Genotype']=="APP"],
        color=cluster_to_use,
        add_outline=False,
        legend_loc="on data",
        legend_fontsize=12,
        legend_fontoutline=2,
        frameon=True,
        palette="tab20",
    )


# sc.pl.pca(adata_microglia,
#          color=cluster_to_use,
#          palette="tab20",
#          )

In [ ]:
samples_ids_sub = adata_microglia.obs['sample'].unique()

# Map all cells
fig, axs = plt.subplots(2,3,figsize=(15, 6))
axs = axs.flatten()
clusters_plot = {"0": "#EE2A96","1": '#3f488aff',"2": 'green', "3":'red', "4":'orange',"5":'black',"6":"purple"
                }

for idx, sample in enumerate(samples_ids_sub):
    adata_sel = adata_microglia[(adata_microglia.obs['sample'] == sample)&(adata_microglia.obs['region_automap_name']=="SCH")]
    for cluster_id in adata_sel.obs[cluster_to_use].unique():
        cluster_data = adata_sel.obs[adata_sel.obs[cluster_to_use] == cluster_id]
        colors = clusters_plot[cluster_id] if cluster_id in clusters_plot else "none" ### for selected clusters in cluster_plot
        # colors= cluster_data['leiden_colors'].unique()[0] ### for all clusters
        axs[idx].scatter(cluster_data['x_centroid'].astype('float'), cluster_data['y_centroid'].astype('float'), color=colors, s=5, label=cluster_id, )
        axs[idx].set_title(f"Sample {sample}")
        # axs[idx].set_ylim(400,1300)
        # axs[idx].set_xlim(4100,5600)
plt.legend()

plt.savefig(f'Gallery/{today}/{celltype_to_subset}_R0_clusterplot.svg', dpi=300, transparent=True, format="svg")

In [ ]:
adata_sel = adata_microglia[(adata_microglia.obs['sample'] == "3160-1")]
cluster_to_use = 'leiden'
clusters_plot = {"1": '#EE2A35',"": '#3f488aff',"": 'green', "":'pink', "":'orange',"":'black',"":"purple"
                }

plt.figure(figsize=(8,5))
for cluster_id in adata_sel.obs[cluster_to_use].unique():
    cluster_data = adata_sel.obs[adata_sel.obs[cluster_to_use] == cluster_id]
    colors = clusters_plot[cluster_id] if cluster_id in clusters_plot else "none" ### for selected clusters in cluster_plot
    colors= cluster_data['leiden_colors'].unique()[0] ### for all clusters
    plt.scatter(cluster_data['x_centroid'].astype('float'), cluster_data['y_centroid'].astype('float'), color=colors, s=1, label=cluster_id)
    # plt.title(f"Sample {samples_ids_sub[4]}")
    # axs[idx].set_ylim(400,1300)
    # axs[idx].set_xlim(4100,5600)
plt.legend()

In [ ]:
### Correlation map of subclusters
cont_tab = pd.crosstab(adata_microglia.obs[cluster_to_use], adata_microglia.obs['mmc:supertype_name'], normalize="index")
cont_tab = cont_tab.loc[:, cont_tab.sum(axis=0) > 0.01] 
plt.figure(figsize=(10,15))
sns.heatmap(cont_tab.T, annot=True, cmap="YlGnBu", fmt=".1f")

In [ ]:
# sc.pl.stacked_violin(adata_microglia, marker_genes, groupby=[condition_to_use,'leiden'], dendrogram=False,)

In [ ]:
# sc.pl.violin(adata_microglia, marker_genes, groupby='leiden'#, order = ['WT','APP'],
#              #log = True,
#              # stripplot = False,
#             )

In [ ]:
adata_microglia.obs['mmc:subclass_name'].value_counts(normalize=True)

In [ ]:
# Obtain cluster-specific differentially expressed genes
cluster_to_use = 'leiden'
# cluster_to_use = "kmeans"
# cluster_to_use = 'Genotype'
adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype('category')
sc.tl.dendrogram(adata_microglia, groupby = cluster_to_use, n_pcs=None, use_rep=None, var_names=None, use_raw=None, cor_method='pearson', linkage_method='complete', optimal_ordering=False, key_added=None)
sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata_microglia, groupby=cluster_to_use, standard_scale="var", n_genes=5)

sc.pl.rank_genes_groups_dotplot(
    adata_microglia,
    n_genes=5,
    values_to_plot="logfoldchanges", cmap='bwr',
    vmin=-4,
    vmax=4,
    save=True
)

In [ ]:
# Obtain cluster-specific differentially expressed genes
# cluster_to_use = 'kmeans'
cluster_to_use = 'Genotype'
adata_microglia.obs[cluster_to_use] = adata_microglia.obs[cluster_to_use].astype(str)
# sc.tl.dendrogram(adata_microglia, groupby = cluster_to_use, n_pcs=None, use_rep=None, var_names=None, use_raw=None, cor_method='pearson', linkage_method='complete', optimal_ordering=False, key_added=None)
sc.tl.rank_genes_groups(adata_microglia, groupby=cluster_to_use, method="wilcoxon", tie_correct = True)
sc.pl.rank_genes_groups_dotplot(adata_microglia, groupby=cluster_to_use, standard_scale="var", n_genes=5)

sc.pl.rank_genes_groups_dotplot(
    adata_microglia,
    n_genes=5,
    values_to_plot="logfoldchanges", cmap='bwr',
    # vmin=-4,
    # vmax=4,
)

In [ ]:
### Extract gene expression per cluster + log fold change + p-value
# cluster_to_use = 'run'
cluster_to_use = 'Genotype'
dfs = []
dfs_filter = []
filters_ = True
filters_dic = {'percentage_pop': 0.10,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.01}

all_groups = adata_microglia.obs['leiden'].unique().astype("category")

for cluster in all_groups:
    adata_microglia_sub = adata_microglia[adata_microglia.obs['leiden'] == cluster]
    sc.pp.calculate_qc_metrics(adata_microglia_sub, percent_top=[20, 50], inplace=True)

    adata_microglia_sub.obs[cluster_to_use] = adata_microglia_sub.obs[cluster_to_use].astype(str)
    #sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
    sc.tl.rank_genes_groups(adata_microglia_sub, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)

    # sc.pl.rank_genes_groups_dotplot(adata_microglia_sub, groupby=cluster_to_use, standard_scale="var",
    #                                 values_to_plot="logfoldchanges", cmap='bwr', n_genes=5)

    dat = pd.DataFrame()
    for i in adata_microglia_sub.obs[cluster_to_use].unique():
        print(f"Cluster {i}")
        dat1 = sc.get.rank_genes_groups_df(adata_microglia_sub, group=i)
        dat1['group'] = i
        dat = pd.concat([dat, dat1])
        dat["mean_count"] = dat["names"].map(dict(zip(adata_microglia_sub.var.index, adata_microglia_sub.var.mean_counts)))

    
    if filters_ == True:
        dat_filter = dat[ ### Choose filters here
        # (dat['pct_nz_group'] > filters_dic['percentage_pop']) & #Percentage of cell expressing the gene
        (dat['pvals_adj']<= filters_dic['pval_adj']) & # adjusted p-value
        (abs(dat['logfoldchanges']) > filters_dic['logfoldchanges']) & # logfoldchange
        (dat['mean_count'] >= filters_dic['mean_count'])
        ]
        dfs_filter.append(dat_filter)

    dfs.append(dat)

## Extract not-normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_processed}/analysis/{name_dir}/foldchanges/sub_clustered_region/"):
    os.makedirs(f"{dir_processed}/analysis/{name_dir}/foldchanges/sub_clustered_region/")


writer = pd.ExcelWriter(f'{dir_processed}/analysis/{name_dir}/foldchanges/sub_clustered_region/DEG_{celltype_to_subset}_no-filter.xlsx', engine='xlsxwriter')
for j in range(0,len(dfs)):
    dfs[j].to_excel(writer, sheet_name=all_groups[j], index=False)
writer.close()

writer = pd.ExcelWriter(f'{dir_processed}/analysis/{name_dir}/foldchanges/sub_clustered_region/DEG_{celltype_to_subset}_filter.xlsx', engine='xlsxwriter')
for j in range(0,len(dfs)):
    dfs_filter[j].to_excel(writer, sheet_name=all_groups[j], index=False)
writer.close()

In [ ]:
dfs_filter[0] = dfs_filter[0][dfs_filter[0]['group'] == 'SD1']
dfs_filter[1] = dfs_filter[1][dfs_filter[1]['group'] == 'SD1']
dfs[0] = dfs[0][dfs[0]['group'] == 'SD1']
dfs[1] = dfs[1][dfs[1]['group'] == 'SD1']

In [ ]:
dfs_filter[1].sort_values('pvals_adj')

In [ ]:
# Obtain cluster-specific differentially expressed genes

adata_micro_0 = adata_microglia[adata_microglia.obs['leiden']=="0"]

cluster_to_use = 'Genotype'
adata_micro_0.obs[cluster_to_use] = adata_micro_0.obs[cluster_to_use].astype("category")
# sc.tl.dendrogram(adata_micro_0, groupby = cluster_to_use, n_pcs=None, use_rep=None, var_names=None, use_raw=None, cor_method='pearson', linkage_method='complete', optimal_ordering=False, key_added=None)
sc.tl.rank_genes_groups(adata_micro_0, groupby=cluster_to_use, method="wilcoxon", tie_correct = True)
sc.pl.rank_genes_groups_dotplot(adata_micro_0, groupby=cluster_to_use, standard_scale="var", n_genes=5)

sc.pl.rank_genes_groups_dotplot(
    adata_micro_0,
    n_genes=5,
    values_to_plot="logfoldchanges", cmap='bwr',
    # vmin=-4,
    # vmax=4,
    save=True
)

In [ ]:
# Obtain cluster-specific differentially expressed genes

adata_micro_0 = adata_microglia[adata_microglia.obs['leiden']=="1"]

cluster_to_use = 'Genotype'
adata_micro_0.obs[cluster_to_use] = adata_micro_0.obs[cluster_to_use].astype("category")
# sc.tl.dendrogram(adata_micro_0, groupby = cluster_to_use, n_pcs=None, use_rep=None, var_names=None, use_raw=None, cor_method='pearson', linkage_method='complete', optimal_ordering=False, key_added=None)
sc.tl.rank_genes_groups(adata_micro_0, groupby=cluster_to_use, method="wilcoxon", tie_correct = True)
sc.pl.rank_genes_groups_dotplot(adata_micro_0, groupby=cluster_to_use, standard_scale="var", n_genes=5)

sc.pl.rank_genes_groups_dotplot(
    adata_micro_0,
    n_genes=5,
    values_to_plot="logfoldchanges", cmap='bwr',
    # vmin=-4,
    # vmax=4,
    save=True
)

In [ ]:
plt.vlines(x=(-0.26,0.26), ymin=10e-24, ymax=1, color = "black", linestyles='dashed')
plt.hlines(y=0.05, xmin=-1, xmax=1, color = "black", linestyles='dashed')
plt.scatter(x=dfs[1]['logfoldchanges'], y = dfs[1]['pvals_adj'], alpha= 0.75, color = "grey", edgecolors=None,s=2)
plt.scatter(x= dfs_filter[1]['logfoldchanges'], y=dfs_filter[1]['pvals_adj'], alpha=1, color = 'red',s=2)
# for idx, gene in enumerate(dfs_filter[1]['names']):
#     plt.text(dfs_filter[1]['logfoldchanges'][idx],  dfs_filter[1]['pvals_adj'][idx], str(gene), color = 'black', fontsize = 12, ha= 'center')
plt.yscale('log')
plt.xlim(-4,4)
plt.gca().invert_yaxis()
plt.xlabel('Log2 Foldchange')
plt.ylabel('Adjusted p-value')
# plt.savefig('Gallery/volcano_plot_SCN.svg')

In [ ]:
from module.misc import genes_list

marker_genes = ['Vip','Vipr1','Vipr2','Adcyap1','Grp', 'Prok1','Prokr2', 'Six3','Six6',"Slc12a5"]
# # marker_genes = genes_list('clock')
# marker_genes = ['Dio2',"Nrn1","Vsir","Fzd1","Il22","Cpox", "Nup98", "Ints11", "Ss18l1", "Cbfb"]
# marker_genes = ["Vip", "Ywhaz", "Ptprn2", "Atp1b2", "Ndrg2"]


In [ ]:
ax = sc.pl.stacked_violin(adata_microglia, ### Can be more useful on subset of the data, otherwise "zero values" greatly change the graph
                         marker_genes, ### marker_genes or individual genes (ex: "Dner")
                         groupby=['leiden','run'],
                         dendrogram=False,
                         log=True,
                         density_norm='count',
                         figsize= (4,2),

                         )

In [ ]:
sc.pl.violin(adata_microglia, marker_genes, groupby='leiden',
             jitter = 0.45,
             # log = True,
             # stripplot = False,
            )

## Subset one region

In [ ]:
adata2.obs['region_automap_name'].unique()

In [ ]:
adata2 = adata.copy()

In [ ]:
import geopandas as gpd
sample_ids = ["3161-1",'3161-2',"3161-3"]

area_list = []
ctx_list = []

for sample_to_map in sample_ids:
    plaque_gpd = gpd.read_file(f'{dir_processed}/coordinates/plaques/{sample_to_map}_plaques_c.geojson')
    plaque_gpd['area'] = plaque_gpd.area
    area_list.append(plaque_gpd['area'].sum())
    ctx_gpd = gpd.read_file(f'{dir_processed}/coordinates/Region_prediction/Xenium-data-coordinates-filtered_{sample_to_map}.geojson')
    ctx_gpd['area'] = ctx_gpd.area
    ctx_list.append(ctx_gpd['area'].sum())
sum(area_list), sum(ctx_list)

In [ ]:
region_to_subset = "CTX"
adata_region = adata2[adata2.obs['region_automap_name'] == region_to_subset]

# sample_ids_ad = ["3161-1",'3161-2',"3161-3"]
# adata_region = adata_region[adata_region.obs['sample'].isin(sample_ids_ad)]

In [ ]:

sc.pp.pca(adata_region, n_comps = 50)
sc.pp.neighbors(adata_region)
sc.tl.umap(adata_region, min_dist = 0.5, spread=2, n_components=5)

# X_pca = adata_region.obsm['X_pca'] 

### Kmeans clustering
### You can choose the number of clusters by uncommenting n_clusters option
# kmeans = KMeans(n_clusters=2,
#                 random_state=0).fit(X_pca) 
# adata_microglia.obs['kmeans'] = kmeans.labels_.astype(str)

sc.tl.leiden(adata_region, resolution = 0.7, flavor = "leidenalg", n_iterations=2)

In [ ]:
cluster_to_use = "leiden"

In [ ]:
from matplotlib.pyplot import rc_context
with rc_context({"figure.figsize": (5, 5)}):
    sc.pl.umap(
        adata_region[adata_region.obs['Genotype']=="WT"],
        color=cluster_to_use,
        add_outline=False,
        legend_loc="on data",
        legend_fontsize=12,
        legend_fontoutline=2,
        frameon=True,
        palette="tab20",
    )

with rc_context({"figure.figsize": (5, 5)}):
    sc.pl.umap(
        adata_region[adata_region.obs['Genotype']=="APP"],
        color=cluster_to_use,
        add_outline=False,
        legend_loc="on data",
        legend_fontsize=12,
        legend_fontoutline=2,
        frameon=True,
        palette="tab20",
    )


In [ ]:
samples_ids_sub = adata_region.obs['sample'].unique()

# Map all cells
fig, axs = plt.subplots(2,3,figsize=(15, 6))
axs = axs.flatten()
clusters_plot = {"0": "#EE2A96","1": '#3f488aff',"2": 'green', "3":'red', "4":'orange',"5":'black',"6":"purple"
                }

for idx, sample in enumerate(samples_ids_sub):
    adata_sel = adata_region[(adata_region.obs['sample'] == sample)]
    for cluster_id in adata_sel.obs[cluster_to_use].unique():
        cluster_data = adata_sel.obs[adata_sel.obs[cluster_to_use] == cluster_id]
        colors = clusters_plot[cluster_id] if cluster_id in clusters_plot else "none" ### for selected clusters in cluster_plot
        # colors= cluster_data['leiden_colors'].unique()[0] ### for all clusters
        axs[idx].scatter(cluster_data['x_centroid'].astype('float'), cluster_data['y_centroid'].astype('float'), color=colors, s=0.05, label=cluster_id, )
        axs[idx].set_title(f"Sample {sample}")
        # axs[idx].set_ylim(400,1300)
        # axs[idx].set_xlim(4100,5600)
plt.legend()
plt.show()

# plt.savefig(f'Gallery/{today}/{celltype_to_subset}_R0_clusterplot.svg', dpi=300, transparent=True, format="svg")

## Plaque cell density

In [ ]:
df_ctx = adata_region.obs.groupby('plaque')['cell_type_final'].value_counts() / (124.84261552709007-1.725070276869839)
df_ctx = df_ctx['No']
df = pd.DataFrame(index=df_ctx.index)
df['CTX'] = df_ctx

In [ ]:
df_plaque = adata_region.obs.groupby('plaque')['cell_type_final'].value_counts() / 1.725070276869839
df_plaque = df_plaque['Yes']
df['plaque'] = df_plaque

In [ ]:
df['Diff'] = df['plaque'] - df['CTX']
df.sort_values(by='plaque', ascending=False, inplace = True)

plt.figure(figsize=(3,5))
plt.barh(y=df.index[0:10],  width = df['CTX'][0:10], height=-0.5, align="edge", label = "outside plaque")
plt.barh(y=df.index[0:10], width = df['plaque'][0:10], height=0.5, align="edge", label ="within plaque")
plt.gca().invert_yaxis()
plt.xticks(rotation=0)
plt.legend()
plt.xlabel('Cell density (#/µm²)')

plt.savefig(f'Gallery/{today}/plaque_cell_density.svg', dpi=300, transparent=True)

In [ ]:
# Generate new numbering base on unique 'cell type'
all_cell_type = adata_region.obs['cell_type_final'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata_region.obs['cell_type_newnum_final'] = adata_region.obs['cell_type_final'].map(mapping_dict)
# mapping_dict

### Generate a color palette for the clusters - to make color stay consistent across samples
adata_region.obs['cell_type_newnum_final'] = adata_region.obs['cell_type_newnum_final'].astype(int)

# Create a palette with a unique color for each cluster
num_clusters = len(adata_region.obs['cell_type_newnum_final'].unique())
palette = sns.color_palette("bright", n_colors=num_clusters)

# Map each 'leiden' value to a color
adata_region.obs['kmeans_colors'] = adata_region.obs['cell_type_newnum_final'].apply(lambda x: palette[x])

# Mapping of clusters
fig, axs = plt.subplots(3,4,figsize=(15, 10))
axs = axs.flatten()
clusters_plot = { 1: 'lightcoral', 11:'black',4:'red',
    # 0: 'orchid', 1: 'forestgreen',2: 'coral', 4:'orange',
    # 3:'red', 5:'blue',6:'cyan',7:'black'
    # 4:'red',0:'black'
}

for idx, sample in enumerate(samples_ids):
    adata_sel = adata_region[(adata_region.obs['sample'] == sample)]
    for cluster_id in adata_sel.obs['cell_type_newnum_final'].unique():
        cluster_data = adata_sel.obs[adata_sel.obs['cell_type_newnum_final'] == cluster_id]
        if len(cluster_data) >= 0:
            colors = clusters_plot[cluster_id] if cluster_id in clusters_plot else "none"
            colors= cluster_data['kmeans_colors'].unique()[0]
            axs[idx].scatter(cluster_data['x_centroid'], cluster_data['y_centroid'], color=colors, s=0.01, label=cluster_data['cell_type_final'].unique()[0])
            axs[idx].set_title(f"Sample {sample}")

plt.legend(markerscale=1, scatterpoints=1000, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)


In [ ]:
# len(adata_region)/1000
# adata_region_notunique.obs.groupby('cell type')['cell_id'].nunique()
# list_to_exclude

In [ ]:
list_to_exclude  = adata_region.obs.groupby('cell_type_final')['cell_id'].nunique() >= 10
list_to_exclude.values
dict_exclude = dict(zip(list_to_exclude.index, list_to_exclude.values))
dict_exclude

adata_region.obs['exclude'] = adata_region.obs['cell_type_final'].map(dict_exclude)

adata_region_notunique = adata_region[adata_region.obs['exclude'] != False]

adata_region_notunique.obs.groupby('cell_type_final')['cell_id'].nunique()

sc.tl.dendrogram(adata_region_notunique, groupby = 'plaque', n_pcs=None, use_rep=None, var_names=None, use_raw=None, cor_method='pearson', linkage_method='complete', optimal_ordering=False, key_added=None)
sc.tl.rank_genes_groups(adata_region_notunique, groupby='plaque', method="wilcoxon", tie_correct = True)


In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_region_notunique, groupby='plaque', standard_scale="var", n_genes=10
                                )

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata_region_notunique,
    n_genes=10,
    values_to_plot="logfoldchanges", cmap='bwr',
    # vmin=-4,
    # vmax=4,
    save=True
)

In [ ]:
### Extract gene expression per cluster + log fold change + p-value
cluster_to_use = 'plaque'

adata_region.obs[cluster_to_use] = adata_region.obs[cluster_to_use].astype(str)
#sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
sc.tl.rank_genes_groups(adata_region, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)

sc.pl.rank_genes_groups_dotplot(adata_region, groupby=cluster_to_use, standard_scale="var", n_genes=5)

dat = pd.DataFrame()
for i in adata_region.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata_region, group=i)
    dat1['group'] = i
    dat = pd.concat([dat, dat1])

### Extract not-normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_processed}/csv/{name_dir}/"):
   os.makedirs(f"{dir_processed}/csv/{name_dir}/")
dat.to_csv(f"{dir_processed}/csv/{name_dir}/{name_dir}_clusters_foldchange_{region_to_subset}-region_plaque.csv")

In [ ]:
celltype_to_subset = "Astro TE"
adata_region_cell = adata_region[adata_region.obs['cell_type_final'] == celltype_to_subset]

In [ ]:
### Extract gene expression per cluster + log fold change + p-value
cluster_to_use = 'plaque'

adata_region_cell.obs[cluster_to_use] = adata_region_cell.obs[cluster_to_use].astype(str)
#sc.tl.dendrogram(adata, groupby = 'L04_newnum_subclassname')
sc.tl.rank_genes_groups(adata_region_cell, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True)

sc.pl.rank_genes_groups_dotplot(adata_region_cell, groupby=cluster_to_use, standard_scale="var", n_genes=5)

dat = pd.DataFrame()
for i in adata_region_cell.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata_region_cell, group=i)
    dat1['group'] = i
    dat = pd.concat([dat, dat1])

### Extract not-normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_processed}/csv/{name_dir}/"):
   os.makedirs(f"{dir_processed}/csv/{name_dir}/")
dat.to_csv(f"{dir_processed}/csv/{name_dir}/{name_dir}_clusters_foldchange_{region_to_subset}-region-{celltype_to_subset}-cell-plaque.csv")

In [ ]:
dat1_filter = dat1[(~dat1['logfoldchanges'].between(-0.26,0.26))
                   & (dat1['pvals_adj'] <= 0.01)
                    & (dat1['pct_nz_group'] > 0.05)
                      ]
dat1_filter.reset_index(inplace=True)

In [ ]:
dat1['Q'] = -np.log10(dat1['pvals_adj'])

dat1_filter['Q'] = -np.log10(dat1_filter['pvals_adj'])

plt.vlines(x=(-0.26,0.26), ymin=0, ymax=max(dat1["Q"]), color = "black", linestyles='dashed', linewidth = 1)
plt.hlines(y=0.05, xmin=-1, xmax=1, color = "black", linestyles='dashed')
plt.scatter(x=dat1['logfoldchanges'], y = dat1['Q'], alpha= 0.75, color = "grey", edgecolors=None,s=5)
plt.scatter(x= dat1_filter['logfoldchanges'], y=dat1_filter['Q'], alpha=1, color = 'red',s=5)
# 
#     
dat_text = dat1_filter[dat1_filter["Q"] > 10].copy()
dat_text = dat_text.reset_index()
texts = []
for idx, gene in enumerate(dat_text['names']):
    texts.append(plt.text(dat_text['logfoldchanges'][idx], dat_text['Q'][idx], str(gene), color = 'black', fontsize = 10, ha= 'center'))
adjust_text(texts, avoid_self=True, pull_threshold =25, arrowprops=dict(arrowstyle="-", color='black', lw=0.5))

# plt.yscale('log')
plt.xlim(-3,4)
# plt.gca().invert_yaxis()
plt.xlabel('Log2 Foldchange')
plt.ylabel('-log Adjusted p-value')
plt.savefig(f'Gallery/{today}/volcano_plot_SCN.svg')

In [ ]:
print(dat1_filter[dat1_filter['logfoldchanges']>0]['names'].values)

# Differential expression analysis

In [12]:
### Extract all cluster compared to all others in a single sheet = markers for each cluster

## Warning : This combine all conditions

cluster_to_use = 'leiden'

sc.tl.rank_genes_groups(adata, groupby=cluster_to_use, method="wilcoxon", tie_correct = True, pts = True,
                            #  layer = 'log1p_norm'
                            )

dat = pd.DataFrame()
for i in adata.obs[cluster_to_use].unique():
    print(f"Cluster {i}")
    dat1 = sc.get.rank_genes_groups_df(adata, group=str(i))
    adata_temp = adata[adata.obs[cluster_to_use]==i]
    sc.pp.calculate_qc_metrics(adata_temp, inplace=True)
    dat1['group'] = i
    dat1["mean_count"] = dat1["names"].map(dict(zip(adata_temp.var.index, adata_temp.var.mean_counts)))
    dat = pd.concat([dat, dat1])

dat.to_csv(f"{dir_processed}/analysis/{name_dir}/{name_dir}_{cluster_to_use}_markergenes.csv")

Cluster 8


IndexError: Positions outside range of features.

In [ ]:
dat.to_csv(f"{dir_processed}/analysis/{name_dir}/{name_dir}_{cluster_to_use}_markergenes.csv")

In [ ]:
dat.sample(10)

## DEG whole brain

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='sample', method="wilcoxon", tie_correct = True, pts = True)

In [ ]:
grp_expe = 'hLiver-cancer'

filters_dict = {'percentage_pop': 0.10,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.01}

for i in adata.obs['sample'].unique():
    # print(f"Cluster {cell_type_to_extract}_{i}")
    dat = sc.get.rank_genes_groups_df(adata, group=grp_expe)
    # sc.pp.calculate_qc_metrics(adata, inplace=True)
    # dat["mean_counts"] = dat["names"].map(dict(zip(adata.var.index, adata.var.mean_counts)))

dat_filter = dat[ ### Choose filters here
        (dat['pct_nz_group'] > filters_dic['percentage_pop']) & #Percentage of cell expressing the gene
        (dat['pvals_adj']<= filters_dict['pval_adj']) & # adjusted p-value
        (abs(dat['logfoldchanges']) > filters_dict['logfoldchanges'])  # logfoldchange
        # (dat['mean_count'] >= filters_dict['mean_counts'])
        ]


In [ ]:
dat_filter

In [ ]:
# ### Extract normalized expression and clusters for individual cells
if not os.path.exists(f"{dir_processed}/analysis/{name_dir}/foldchanges/wholesection"):
   os.makedirs(f"{dir_processed}/analysis/{name_dir}/foldchanges/wholesection")
dat.to_csv(f"{dir_processed}/analysis/{name_dir}/foldchanges/wholesection/{name_dir}_foldchange_wholesection.csv")
dat_filter.to_csv(f"{dir_processed}/analysis/{name_dir}/foldchanges/wholesection/{name_dir}_foldchange_wholesection_filter.csv")

## DEG (one condition)

In [ ]:
filters_dict = {'percentage_pop': 0.1,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.2}

filters_ = True


In [10]:
adata.obs['sample'].unique()

['hLiver-cancer', 'hLiver-nondiseased']
Categories (2, object): ['hLiver-cancer', 'hLiver-nondiseased']

In [8]:
df_all = da.DEG_one_condition(adata,
                        name_dir,
                        cluster_to_use = "cell_type_final",
                        grp_ctrl = "WT",
                        group_col = "Genotype",
                        filters_bool= True, ### Will always save results without filters
                        filters_dict = filters_dict
                        )

[========================================================================] 100%


Extraction done


In [9]:
from pathlib import Path
import anndata as ad
import holoviews as hv
import panel as pn
import hvplot.pandas    # noqa
import numpy as np
import pooch

import scanpy as sc

import hv_anndata
from hv_anndata import *

hv_anndata.register()
hv.extension("bokeh")
pn.extension()
pn.config.throttled = True

In [12]:
test = 'APP'
control = 'WT'

for idx in range(len(df_all)):
    if control in df_all[idx]["group"].unique():
        df_all[idx] = df_all[idx][df_all[idx]['group'] == test]

In [13]:
da.interactive_volcano_plot(df_all,
                            'wilcoxon',
                            "TH_Glut",
                            'hLiver-nondiseased',
                            'hLiver-cancer')

TypeError: list indices must be integers or slices, not str

## DEG (2 conditions)

In [ ]:
from module.dataviz_analysis import DEG_two_conditions

filters_dict = {'percentage_pop': 0.1,
               'pval_adj' : 0.05,
               'logfoldchanges' : 0.26,
               'mean_count' : 0.2}

filters_ = True

In [ ]:
dfs = DEG_two_conditions(adata,
                        name_dir,
                        cluster_to_use_1 = "region_automap_name",
                        cluster_to_use_2= "leiden",
                        grp_ctrl = "WT",
                        group_col = "Genotype",
                        filters_bool= True, ### Will always save results without filters
                        filters_dict = filters_dict
                        )

## Deseq2

In [ ]:
import module.dataviz_analysis as da

filters_dict = {'logfoldchanges' : 0.26,
                'padj' : 0.05}

cluster_to_use = "leiden"

ddf, ddf_filter, list_celltypes_sheet = da.deseq2_one_condition(adata,
                                                             name_dir,
                                                             cluster_to_use = cluster_to_use,
                                                             group_col="sample",
                                                             filters_bool = True,
                                                             filters_dict = filters_dict,
                                                             pseudoreplicates = 3,
                                                             )

Analysis Done
13 analyzed.
0 ignored.


In [2]:
da.interactive_volcano_plot(ddf,'DeSeq2',0,'hLiver-nondiseased','hLiver-cancer')

AttributeError: module 'module.dataviz_analysis' has no attribute 'interactive_volcano_plot'

# Other outputs

## Extract cell population comparison

In [ ]:
adata_region = adata[adata.obs['region_automap_name'] == 'SCH']

In [ ]:
# adata_region.obs.groupby('Genotype', as_index=False)['cell_type_final'].value_counts(normalize=False)
df_t = adata_region.obs.groupby('sample', as_index=False)[['Genotype','cell_type_final']].value_counts(normalize=False)
adata_region.obs.value_counts(['Genotype','cell_type_final',], normalize=False)


In [ ]:
df_t.to_csv(f'csv/{name_dir}/{name_dir}_SCH_cellpop.csv')

## Extract cell population

In [ ]:
df.groupby(['Genotype','cell_type_final'])['cell_type_final'].value_counts().to_csv(f'{dir_processed}/analysis/{name_dir}/summary_cell_number.csv')

In [ ]:
cell_nb = pd.read_csv(f'{dir_processed}/analysis/{name_dir}/summary_cell_number.csv')
cell_nb_SD = cell_nb[cell_nb['run'] == 'SD1']
cell_nb_circa = cell_nb[cell_nb['run'] == 'circa4']
cell_nb_circa.head(1)

In [ ]:
cell_nb_circa[cell_nb_circa['cell_type_final'] == "Tanycyte"]

In [ ]:
plt.figure(figsize=(5,15))
plt.barh(cell_nb_circa['cell_type_final'], cell_nb_circa['count'], height = 0.4, align = "edge", label = "NS")
plt.barh(cell_nb_SD['cell_type_final'], cell_nb_SD['count'], height = -0.4, align = "edge", label = "SF")
plt.xscale('log')
plt.legend()
plt.ylim(-0.5,78.5)

## Extract cell pop per region

In [ ]:
grouped = df.groupby(['run','sample','region_automap_name'])['cell_type_final'].value_counts(dropna=True)
grouped.to_csv(f'{dir_processed}/analysis/{name_dir}/cell_count.csv')
# df_cellpop = pd.read_csv('data/test_cell_count.csv')

In [ ]:
df.groupby(['sample'])['region_automap_name'].value_counts().to_csv('data/region_count_circa.csv')

## Gene expression

In [ ]:
run_ = 'SD1'
param = 'run'
df_sub = df[df[param] == run_]

from module.misc import genes_list

gene_list = genes_list('panel_5k_circa')

### celltype gene expression
ddf = df_sub.groupby('cell_type_final')[gene_list].mean()
expr_frac = df_sub[gene_list].gt(0).groupby(df_sub['cell_type_final']).mean()
expr_frac = expr_frac.T
ddf = ddf.T
celltypes = list(ddf.keys())
# celltypes.remove("Tanycyte")
NS_dict = {}

path_to_save = f"{dir_processed}/analysis/{name_dir}/gene_list_{run_}"
if not os.path.exists(path_to_save):
    os.makedirs(path_to_save)

for cell in celltypes:
    print(cell)
    ddf_temp = dict(zip(ddf[cell].index, ddf[cell].values))
    ddf_temp = {key:value for key, value in ddf_temp.items() if value > 0.2}
    ddf_temp = set([key for key in ddf_temp.keys()])

    expr_frac_temp = dict(zip(expr_frac[cell].index, expr_frac[cell].values))
    expr_frac_temp = {key:value for key, value in expr_frac_temp.items() if value >= 0.1}
    expr_frac_temp = set([key for key in expr_frac_temp.keys()])
    
    gene_expressed = list(ddf_temp.intersection(expr_frac_temp))
    NS_dict[cell] = list(gene_expressed)
    gene_expressed = pd.Series(gene_expressed)
    gene_expressed.to_csv(f"{path_to_save}/{cell}.csv")
    clear_output()

In [ ]:
len(gene_expressed)

In [ ]:
run_ = 'SD1'
param = 'run'
df_sub = df[df[param] == run_]

from module.misc import genes_list

gene_list = genes_list('panel_5k_circa')


### celltype gene expression
ddf = df_sub.groupby('cell_type_final')[gene_list].mean()
ddf = ddf.T
expr_frac = df_sub[gene_list].gt(0).groupby(df_sub['cell_type_final']).mean()
expr_frac = expr_frac.T

SF_dict = {}

path_to_save = f"/media/volume/volume_spatial/hugo/R/data/gene_list_{run_}/"
if not os.path.exists(path_to_save):
    os.makedirs(path_to_save)

for cell in celltypes:
    print(cell)
    ddf_temp = dict(zip(ddf[cell].index, ddf[cell].values))
    ddf_temp = {key:value for key, value in ddf_temp.items() if value > 0.2}
    ddf_temp = set([key for key in ddf_temp.keys()])

    expr_frac_temp = dict(zip(expr_frac[cell].index, expr_frac[cell].values))
    expr_frac_temp = {key:value for key, value in expr_frac_temp.items() if value >= 0.1}
    expr_frac_temp = set([key for key in expr_frac_temp.keys()])
    
    gene_expressed = list(ddf_temp.intersection(expr_frac_temp))
    SF_dict[cell] = gene_expressed
    gene_expressed = pd.Series(gene_expressed)
    gene_expressed.to_csv(f"{path_to_save}/{cell}.csv")
    clear_output()

In [ ]:
dddf = pd.DataFrame(index=celltypes)
set_all = set()
set_all_common = set()
set_common_ = set()
for cell in celltypes:
    set_common_ = set()
    set_NS = set(NS_dict[cell])
    set_SF = set(SF_dict[cell])

    set_common = set_NS.intersection(set_SF)
    if len(set_NS) > len(set_SF):
        set_diff = set_NS.difference(set_SF)
    else:
        set_diff = set_SF.difference(set_NS)
    
    dddf.loc[cell,'set_NS'] = len(set_NS)
    dddf.loc[cell,'set_SF'] = len(set_SF)
    dddf.loc[cell,'set_common'] = len(set_common)
    dddf.loc[cell,'set_diff'] = len(set_diff)
    set_all.update(set_NS,set_SF)
    set_common_.update(set_NS,set_SF)
    if len(set_all_common) == 0:
        set_all_common = set_common_
    set_all_common = set_all_common.intersection(set_common_)


In [ ]:
cg_list = genes_list('clock')
count = 0
for gene in cg_list:
    if gene in set_all_common:
        count+=1
    else:
        print(gene)
count 


In [ ]:
from module.misc import genes_list

panel_list = genes_list("clock")
df_genes = pd.DataFrame(index= panel_list)

for gene in panel_list:
    count = 0

    for cell in celltypes:
        if gene in NS_dict[cell]:
            count += 1
        elif gene in SF_dict[cell]:
            count += 1
    df_genes.loc[gene, 'count'] = count

In [ ]:
# df_genes.sort_values(by='count', ascending = True)
df_genes['count'].value_counts()
# df_genes['count'].hist()
# print(*df_genes[df_genes['count']==1].index, sep = ',')
# df_genes.loc['Tlr1','count']

In [ ]:
dddf_gene = pd.DataFrame(index = df_genes[df_genes['count'] == 1].index)
dddf_gene['celltype'] = "0"

for gene in df_genes[df_genes['count'] == 1].index:
    for key in celltypes:
        if gene in NS_dict[key]:
            # print(key)
            dddf_gene.loc[gene,'celltype'] = key
        elif gene in SF_dict[key]:
            dddf_gene.loc[gene,'celltype'] = key

In [ ]:
# dddf_gene['celltype'].value_counts()
dddf_gene[dddf_gene['celltype']=="Microglia"]

In [ ]:
dddf_gene[dddf_gene['celltype']=='OB STR CTX IMN']

In [ ]:
### cellclass gene expression

ddf = df_sub.groupby('cell_class')[gene_list].mean()
ddf = ddf.T

path_to_save = f"/media/volume/volume_spatial/hugo/R/data/gene_list_{run_}/cellclass/"
if not os.path.exists(path_to_save):
    os.makedirs(path_to_save)

for cell in ddf.keys():
    print(cell)
    ddf_temp = dict(zip(ddf[cell].index, ddf[cell].values))
    ddf_temp = {key:value for key, value in ddf_temp.items() if value > 0.01}
    gene_expressed = [key for key in ddf_temp.keys()]
    gene_expressed = pd.Series(gene_expressed)
    gene_expressed.to_csv(f"{path_to_save}/{cell}.csv")
    clear_output()

In [ ]:
### Neurotransmitter gene expression

ddf = df_sub.groupby('Neurotransmitter')[gene_list].mean()
ddf = ddf.T

path_to_save = f"/media/volume/volume_spatial/hugo/R/data/gene_list_{run_}/Neurotransmitter/"
if not os.path.exists(path_to_save):
    os.makedirs(path_to_save)

for cell in ddf.keys():
    print(cell)
    ddf_temp = dict(zip(ddf[cell].index, ddf[cell].values))
    ddf_temp = {key:value for key, value in ddf_temp.items() if value > 0.01}
    gene_expressed = [key for key in ddf_temp.keys()]
    gene_expressed = pd.Series(gene_expressed)
    gene_expressed.to_csv(f"{path_to_save}/{cell}.csv")
    clear_output()

In [ ]:
run_ = "SD1"
param = 'run'
df_sub = df[df[param] == run_]

from module.misc import genes_list

gene_list = genes_list('panel_5k_circa')

In [ ]:
### celltype gene expression in regions

for region in df_sub['region_automap_name'].unique():
    ddf_sub = df_sub[df_sub['region_automap_name'] == region]

    ddf = df_sub.groupby('cell_type_final')[gene_list].mean()
    ddf = ddf.T

    expr_frac = df_sub[gene_list].gt(0).groupby(df_sub['cell_type_final']).mean()
    expr_frac = expr_frac.T

    path_to_save = f"../R/data/gene_list_{run_}/{region}"
    if not os.path.exists(path_to_save):
        os.makedirs(path_to_save)

    for cell in ddf.keys():
        ddf_temp = dict(zip(ddf[cell].index, ddf[cell].values))
        ddf_temp = {key:value for key, value in ddf_temp.items() if value > 0.2}
        ddf_temp = set([key for key in ddf_temp.keys()])

        expr_frac_temp = dict(zip(expr_frac[cell].index, expr_frac[cell].values))
        expr_frac_temp = {key:value for key, value in expr_frac_temp.items() if value >= 0.1}
        expr_frac_temp = set([key for key in expr_frac_temp.keys()])
        
        gene_expressed = list(ddf_temp.intersection(expr_frac_temp))
        # SF_dict[cell] = gene_expressed
        gene_expressed = pd.Series(gene_expressed)
        gene_expressed.to_csv(f"{path_to_save}/{cell}.csv")

In [ ]:
import os
run_ = 'SD1'
directory_expressed_genes = f"{dir_processed}/analysis/{name_dir}/gene_list_{run_}"
all_files = os.listdir(directory_expressed_genes)
all_files = [file for file in all_files if file.split('.')[-1]=="csv"]

cells = []
expressed_genes = [] 

for filename in all_files:
    celltype = filename.split('.')[0]
    celllist = pd.read_csv(f'{directory_expressed_genes}/{filename}')
    cells.append(celltype)   
    expressed_genes.append(len(celllist))

len(cells),len(expressed_genes)

df_genlist = pd.DataFrame(data = {"Celltype" : cells,
                                  "Count" : expressed_genes})

df_genlist.to_csv(f'{dir_processed}/analysis/{name_dir}/{run_}-genexpression-summary.csv')

In [ ]:
df_genlist

In [ ]:
marker_genes = ["Aldh1a2",
"Col1a1",
"Col6a1",
"Cyp1b1",
"Dcn",
"Fmod",
"Gjb2",
"Igf2",
"Pdgfra",
"Ror1",
"Slc13a4",
"Spp1",
]

marker_genes = ["Acta2",
"Ano1",
"Arhgap6",
"Carmn",
"Cspg4",
"Fos",
"Gucy1a1",
"Inpp4b",
"Nr2f2",
"Pip5k1b",
"Plekha2",
"Pln",
"Sncg",
"Sntb1",
]

In [ ]:
gene='Gfap'
df[(df['cell_type_final']=='Astro_NT')].groupby('Genotype')[gene].mean()#hist(bins=100,legend=True, alpha=0.75)
# for gene in marker_genes:
#     print(gene, df[(df['cell_type_final']=='Pericyte')&(df[gene]>0)].groupby('Genotype')[gene].quantile(0.05))

In [ ]:
df[(df[gene]>0.01)].groupby('Genotype')[gene].hist(bins=100,legend=True, alpha=0.75)

In [ ]:
adata.obs['transcript_density'] = adata.obs['total_counts'] / adata.obs['cell_area']

In [ ]:
adata[adata.obs['cell_type_final']=="STR_D2_Gaba"].obs['transcript_density'].hist(bins=100, legend=True)

In [ ]:
adata.obs.groupby('circascore')['transcript_density'].mean().plot(title='transcript density')

In [ ]:
adata.obs.groupby('circascore')['cell_area'].mean().plot(title='cell area')

In [ ]:
adata.obs.groupby('circascore')['total_counts'].mean().plot(title='total_counts')

In [ ]:
adata.obs.groupby('circascore')['n_genes_by_counts'].mean().plot(title='n_genes_by_counts')

In [ ]:
plt.scatter(adata.obs['cell_area'], adata.obs['n_genes_by_counts'], s=0.1)
plt.xlabel('cell_area')
plt.ylabel('n_genes_by_counts')

In [ ]:
plt.figure(figsize=(15,15))
plt.scatter(adata.obs['cell_area'], adata.obs['transcript_density'], s=0.1)
plt.xlabel('cell_area')
plt.ylabel('transcript_density')

In [ ]:
plt.figure(figsize=(15,15))
plt.scatter(adata.obs['n_genes_by_counts'], adata.obs['transcript_density'], s=0.1)
plt.xlabel('n_genes_by_counts')
plt.ylabel('transcript_density')

In [ ]:
plt.figure(figsize=(15,15))
plt.scatter(adata.obs['cell_area'], adata.obs['total_counts'], s=0.1)
plt.xlabel('cell_area')
plt.ylabel('total_counts')

In [ ]:
adata.obs['mmc:subclass_name'].value_counts()

In [ ]:
df_sub['Genotype']

In [ ]:
ddf = df_sub.groupby('cell_type_final')[gene_list].mean()
expr_frac = df_sub[gene_list].gt(0).groupby(df_sub['cell_type_final']).mean()


In [ ]:
for celltype in df.cell_type_final.unique():
    plt.scatter(x= expr_frac.T[celltype], y=ddf.T[celltype], s=1)



In [ ]:
dict_temp = dict(zip(adata.obs["cell_type_final"], adata.obs["cell_type_final"]))

In [ ]:
dict_temp['Astro_NT'] = 'Astrocyte'
dict_temp['Astro_TE'] = 'Astrocyte'


In [ ]:
adata.obs["cell_type_final_bis"] = adata.obs["cell_type_final"].map(dict_temp)

In [ ]:
adata.obs["cell_type_final_bis"].sample(10)

In [ ]:
adata.obs[adata.obs['region_automap_name']=="SCH"].groupby('Genotype')['cell_type_final'].value_counts(normalize=True)

In [ ]:
adata.obs[(adata.obs['region_automap_name']=="SCH")]['mmc:subclass_name'].value_counts()